# Bài 18: Không gian màu (Color Spaces)

Các bài trước đã chuyển ảnh màu sang ảnh xám rồi đi tiếp. Bài này xem xét phép chuyển đổi đó thực sự làm gì, và các cách khác để biểu diễn màu ngoài RGB, mỗi cách phù hợp với một nhiệm vụ khác nhau: **HSV** (tách màu sắc khỏi độ sáng), **YCbCr** (tách độ chói (luminance) khỏi độ màu (chrominance), nền tảng của nén video ở Bài 17), và **L\*a\*b\*** (được thiết kế sao cho khoảng cách khớp với cảm nhận của con người).

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['sheepdog.jpg']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Ảnh xám đã là một lựa chọn không gian màu (Grayscale is already a color-space choice)
Cách đơn giản nhất để chuyển ảnh màu sang ảnh xám là lấy trung bình ba kênh:

$$Y = (R + G + B)/3$$

Dù cách này tạm dùng được, hệ thị giác con người thực ra nhạy với màu lục hơn các màu khác. Do đó, tốt hơn là đánh trọng số cho màu lục nhiều hơn:

$$Y = (R + 2G + B)/4$$

Một cách còn tốt hơn là dùng các trọng số rút ra từ thí nghiệm thực tế trên người, dẫn đến công thức nổi tiếng được dùng trong `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)`:

$$Y = 0.299 R + 0.587 G + 0.114 B$$

Các trọng số này đến từ độ nhạy độ chói của con người: mắt nhạy với màu lục hơn nhiều so với màu lam, nên màu lục và màu lam có cùng cường độ *không* nên ánh xạ về cùng một mức xám, dù phép lấy trung bình kênh ngây thơ sẽ đối xử với chúng như nhau.

### Tại sao công thức này đã lỗi thời (dù được dùng rộng rãi, và vẫn hữu ích) (Why this formula is outdated (even though widely used, and still useful))
Có hai điều ẩn trong công thức đó đáng để biết:

1. **Màu cơ bản cũ.** Các trọng số $0.299, 0.587, 0.114$ đến từ tiêu chuẩn NTSC năm 1953 (Rec. 601), được suy ra từ các chất phát quang (phosphor) đỏ/lục/lam cụ thể dùng trong tivi màu CRT thời kỳ đầu. Màn hình LED hiện đại (và gần như mọi thứ khác được sản xuất từ đầu những năm 2000) dùng các màu cơ bản bão hòa hơn, nên các trọng số chính xác về mặt đo màu cho màn hình hiện đại thực ra là $Y = 0.2126R + 0.7152G + 0.0722B$ (Rec. 709/sRGB): đánh trọng số cho màu lục *nhiều hơn* và cho màu đỏ *ít hơn* một cách đáng kể. `cv2.cvtColor(..., COLOR_BGR2GRAY)` vẫn dùng các trọng số Rec. 601 cũ, chủ yếu để tương thích ngược với hàng chục năm code đã có.
2. **Giả định ánh sáng tuyến tính.** Công thức tổng có trọng số chỉ cho ra độ chói thực sự khi $R$, $G$, $B$ là các giá trị ánh sáng *tuyến tính*. Nhưng như ta sẽ thấy ở Bài 26, các giá trị điểm ảnh được lưu trữ đã được *mã hóa gamma*, không tuyến tính. `cv2.cvtColor` áp dụng các trọng số trực tiếp lên các giá trị 8-bit đã mã hóa thô, không giải mã gamma trước &mdash; nhanh, và đủ gần cho hầu hết mục đích thị giác máy tính, nhưng không phải độ chói chính xác về mặt vật lý mà một ứng dụng hiệu chỉnh màn hình hay trắc quang (photometry) cần. (Để thấy vấn đề, hãy tạo một ảnh lam thuần với RGB=(0,0,255), rồi chuyển đổi bằng OpenCV để nhận giá trị 29 ở mọi nơi &mdash; giá trị này sẽ hiển thị gần như tối hoàn toàn một cách sai lệch, dù màu lam bão hòa hoàn toàn là một màu khá sáng đối với mắt.)

In [ ]:
weights_601 = np.array([0.114, 0.587, 0.299])   # BGR order, what cv2 actually uses (Rec. 601)
weights_709 = np.array([0.0722, 0.7152, 0.2126])  # BGR order, modern primaries (Rec. 709 / sRGB)

for name, bgr in [('pure red', (0, 0, 255)), ('pure green', (0, 255, 0))]:
    bgr = np.array(bgr, dtype=np.float64)
    y_601 = bgr @ weights_601
    y_709 = bgr @ weights_709
    print(f'{name:>10}: Rec. 601 (cv2) Y = {y_601:6.1f}    Rec. 709 (modern) Y = {y_709:6.1f}')

In [ ]:
pure_green = np.full((10, 10, 3), (0, 255, 0), dtype=np.uint8)   # BGR
pure_blue = np.full((10, 10, 3), (255, 0, 0), dtype=np.uint8)

for name, patch in [('green', pure_green), ('blue', pure_blue)]:
    weighted = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)[0, 0]
    naive = patch[0, 0].astype(np.float64).mean()
    print(f'{name:>6}: cv2 (luminance-weighted) gray = {weighted:>3},   naive average = {naive:.0f}')

Phép lấy trung bình ngây thơ hoàn toàn không phân biệt được màu lục và màu lam (cả hai đều cho 85); phiên bản có trọng số theo cảm nhận báo cáo đúng rằng màu lục sáng hơn nhiều (150) so với màu lam (29) ở cùng cường độ kênh.

## Các kênh RGB tương quan cao với nhau (RGB channels are highly correlated)
Trong hầu hết ảnh thật, biến thiên chiếu sáng/đổ bóng chiếm ưu thế: một bề mặt sáng lên hoặc tối đi một cách tổng thể, co giãn cả ba kênh cùng nhau. Điều đó có nghĩa là R, G và B cuối cùng tương quan mạnh với nhau &mdash; không phải là một cách hiệu quả hay tiện lợi lắm để tách "đây là màu gì" khỏi "nó được chiếu sáng đến đâu".

In [ ]:
rng = np.random.default_rng(0)
shading = rng.uniform(0.3, 1.0, (150, 150))  # a smoothly varying "lighting" field
shaded_img = np.stack([shading * 200, shading * 150, shading * 100], axis=-1).astype(np.uint8)  # BGR

shaded_img = cv2.imread('../img/sheepdog.jpg')
r = shaded_img[:, :, 2].ravel().astype(np.float64)
g = shaded_img[:, :, 1].ravel().astype(np.float64)
print(f'correlation between R and G channels: {np.corrcoef(r, g)[0, 1]:.4f}')

img_h, img_w = shaded_img.shape[:2]
fig, axes = plt.subplots(1, 2, figsize=(4 * img_w / img_h + 4, 4),
                          gridspec_kw={'width_ratios': [img_w / img_h, 1]})
axes[0].imshow(cv2.cvtColor(shaded_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Image')
axes[0].axis('off')
axes[1].scatter(r, g, s=2, alpha=0.3)
axes[1].set_xlabel('R'); axes[1].set_ylabel('G')
axes[1].set_title('R and G are highly correlated')
axes[1].set_xlim(0, 255)
axes[1].set_ylim(0, 255)
axes[1].set_aspect('equal', adjustable='box')
plt.show()

## HSV: tách màu sắc khỏi độ sáng (HSV: separating color from brightness)
HSV (Hue — sắc độ, Saturation — độ bão hòa, Value — giá trị/độ sáng) tham số hóa lại màu sao cho Hue nắm bắt *màu nào* (không phụ thuộc vào độ sáng hay độ nhạt của nó), Saturation nắm bắt độ rực/độ thuần của màu, và Value chỉ nắm bắt độ sáng. Điều này khiến phân đoạn dựa trên màu chống chịu với ánh sáng tốt hơn hẳn so với phân ngưỡng trực tiếp trong RGB.

Ảnh động này cho thấy một điểm di chuyển đồng thời trong hai không gian 3D: khối lập phương RGB và hình trụ HSV. Đầu tiên nó quét qua mọi sắc độ ở độ bão hòa và giá trị tối đa, vẽ theo vành ngoài của cả hai hình; sau đó nó tối dần về màu đen ở một sắc độ cố định &mdash; một cú rơi thẳng đứng trong HSV, nhưng là một đường thẳng tiến về gốc tọa độ trong RGB. Điều này minh họa cách HSV tách "màu nào" khỏi "sáng đến đâu" trong khi RGB thì không.

<img src="../animations/rgb_hsv_cube_demo.gif" alt="Ảnh động so sánh quỹ đạo của một màu trong khối lập phương RGB với hình trụ HSV, quét sắc độ quanh vành rồi hạ giá trị thẳng xuống" style="display: block; margin: 0 auto; max-width: 680px;">

In [ ]:
size = 200
flat_img = np.zeros((size, size, 3), dtype=np.uint8)
cv2.circle(flat_img, (100, 100), 70, (0, 100, 255), -1)  # an orange disk, BGR

yy, xx = np.mgrid[0:size, 0:size]
lighting_gradient = (0.25 + 0.9 * (xx / size))  # dark on the left, bright on the right
shaded_disk = np.clip(flat_img.astype(np.float64) * lighting_gradient[..., None], 0, 255).astype(np.uint8)

hsv = cv2.cvtColor(shaded_disk, cv2.COLOR_BGR2HSV)
hue = hsv[:, :, 0]

mask_hsv = (hue > 5) & (hue < 25)                                              # hue range only
mask_rgb = ((shaded_disk[:, :, 2] > 150) & (shaded_disk[:, :, 1] > 50) &
            (shaded_disk[:, :, 1] < 180) & (shaded_disk[:, :, 0] < 80))        # fixed RGB range
true_mask = (xx - 100)**2 + (yy - 100)**2 <= 70**2

def iou(a, b):
    return (a & b).sum() / (a | b).sum()

print(f'IoU vs. true disk, HSV hue threshold: {iou(mask_hsv, true_mask):.3f}')
print(f'IoU vs. true disk, fixed RGB range:   {iou(mask_rgb, true_mask):.3f}')

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
axes[0].imshow(cv2.cvtColor(shaded_disk, cv2.COLOR_BGR2RGB))
axes[0].set_title('Shaded disk (left dark, right bright)')
axes[1].imshow(mask_hsv, cmap='gray')
axes[1].set_title('HSV hue threshold')
axes[2].imshow(mask_rgb, cmap='gray')
axes[2].set_title('Fixed RGB range')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Phân ngưỡng theo sắc độ khôi phục hoàn hảo hình đĩa bất kể dải chuyển sáng trên nó, trong khi một khoảng RGB cố định bỏ sót hoàn toàn phía bị tối &mdash; chính là sự mong manh mà phân đoạn màu thuần RGB gặp phải ngay khi ánh sáng không đồng đều hoàn hảo.

Ảnh động này quét độ dốc của dải chuyển sáng trên hình đĩa bị đổ bóng (từ chiếu sáng đều đến mức như ở trên), tính lại cả hai mặt nạ ngay trong mỗi khung hình. Ngưỡng sắc độ HSV (xanh lục) bám chắc lấy hình đĩa suốt quá trình, trong khi khoảng RGB cố định bị bào mòn thấy rõ khi dải chuyển sáng dốc hơn.

<img src="../animations/hue_threshold_robustness_demo.gif" alt="Ảnh động so sánh phân ngưỡng sắc độ HSV với một khoảng RGB cố định khi dải chuyển sáng trên hình đĩa dốc dần, cho thấy mặt nạ RGB bị bào mòn trong khi mặt nạ HSV giữ ổn định" style="display: block; margin: 0 auto; max-width: 820px;">

## YCbCr: luma và chroma, và tại sao video nén màu mạnh tay hơn (YCbCr: luma and chroma, and why video compresses color more aggressively)
YCbCr (được JPEG và gần như mọi codec video dùng bên trong) tách ảnh thành **luma** ($Y$, gần giống độ sáng) và hai kênh **chroma** ($C_b, C_r$, gần giống "lam đến đâu" và "đỏ đến đâu"). Hệ thị giác con người phân giải chi tiết không gian mịn trong luma tốt hơn nhiều so với trong chroma &mdash; đây là cơ sở sinh học cho **lấy mẫu thưa độ màu (chroma subsampling)** (lưu chroma ở độ phân giải thấp hơn luma, ví dụ định dạng "4:2:0" phổ biến của video), một trong những lợi thế nén miễn phí được dùng xuyên suốt quy trình JPEG ở Bài 17.

In [ ]:
textured = np.zeros((200, 200, 3), dtype=np.uint8)
cv2.rectangle(textured, (20, 20), (90, 180), (255, 120, 30), -1)
cv2.circle(textured, (140, 100), 50, (30, 200, 255), -1)
gradient_x = np.mgrid[0:200, 0:200][1]
textured[:, :, 0] = np.clip(textured[:, :, 0].astype(int) + (gradient_x // 4) % 50, 0, 255)
textured = np.clip(textured.astype(np.float64) + rng.normal(0, 5, textured.shape), 0, 255).astype(np.uint8)

Y, Cr, Cb = cv2.split(cv2.cvtColor(textured, cv2.COLOR_BGR2YCrCb))

def subsample_then_upsample(channel, factor=8):
    small = cv2.resize(channel, (channel.shape[1] // factor, channel.shape[0] // factor), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (channel.shape[1], channel.shape[0]), interpolation=cv2.INTER_LINEAR)

chroma_degraded = cv2.cvtColor(cv2.merge([Y, subsample_then_upsample(Cr), subsample_then_upsample(Cb)]),
                                cv2.COLOR_YCrCb2BGR)
luma_degraded = cv2.cvtColor(cv2.merge([subsample_then_upsample(Y), Cr, Cb]), cv2.COLOR_YCrCb2BGR)

error_chroma = np.abs(chroma_degraded.astype(int) - textured.astype(int)).mean()
error_luma = np.abs(luma_degraded.astype(int) - textured.astype(int)).mean()
print(f'mean abs error, chroma degraded 8x: {error_chroma:.2f}')
print(f'mean abs error, luma degraded 8x:   {error_luma:.2f}')

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [textured, chroma_degraded, luma_degraded],
                          ['Original', 'Chroma subsampled 8x\n(less visible loss)', 'Luma subsampled 8x\n(more visible loss)']):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Loại bỏ cùng một lượng độ phân giải gây ra sai số ít hơn đáng kể ở chroma so với ở luma &mdash; cùng một mức mất chi tiết đơn giản là ít gây khó chịu hơn khi nó xảy ra với màu sắc thay vì độ sáng, và đó chính là sự đánh đổi mà lấy mẫu thưa độ màu khai thác.

## L\*a\*b\*: một không gian đồng đều về cảm nhận (L\*a\*b\*: a perceptually uniform space)
Khoảng cách Euclid trong RGB là một đại diện tồi cho việc hai màu thực sự *trông* khác nhau đến đâu: các khoảng cách RGB bằng nhau có thể tương ứng với những khác biệt cảm nhận chênh lệch rất lớn, tùy thuộc vào vị trí của chúng trong không gian màu. CIELAB được thiết kế một cách có chủ đích sao cho khoảng cách Euclid giữa hai màu L\*a\*b\* xấp xỉ khác biệt cảm nhận nhất quán hơn nhiều &mdash; hữu ích cho các thước đo chất lượng dựa trên màu, phân cụm và so khớp.

In [ ]:
target_rgb_distance = 30.0
n_samples = 2000

c1 = rng.uniform(0, 255, (n_samples, 3))
direction = rng.normal(size=(n_samples, 3))
direction /= np.linalg.norm(direction, axis=1, keepdims=True)
c2 = c1 + direction * target_rgb_distance

valid = np.all((c2 >= 0) & (c2 <= 255), axis=1)
c1, c2 = c1[valid].astype(np.uint8), c2[valid].astype(np.uint8)

lab1 = cv2.cvtColor(c1.reshape(1, -1, 3), cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float64)
lab2 = cv2.cvtColor(c2.reshape(1, -1, 3), cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float64)
lab_distances = np.linalg.norm(lab1 - lab2, axis=1)

print(f'RGB distance held fixed at exactly {target_rgb_distance}')
print(f'resulting L*a*b* distances: min={lab_distances.min():.1f}, '
      f'median={np.median(lab_distances):.1f}, max={lab_distances.max():.1f}')

plt.hist(lab_distances, bins=40)
plt.xlabel('L*a*b* distance')
plt.ylabel('count')
plt.title(f'Perceptual (L*a*b*) distance for {n_samples} color pairs,\nall with the SAME RGB distance ({target_rgb_distance})')
plt.show()

Mỗi cặp trong số này đều cách nhau *chính xác* cùng một khoảng trong RGB &mdash; vậy mà khác biệt cảm nhận (L\*a\*b\*) tương ứng dao động từ hầu như không nhận ra đến hơn 15 lần như thế, chỉ phụ thuộc vào việc cặp màu nằm *ở đâu* trong không gian màu. Bất kỳ thuật toán nào dùng khoảng cách RGB thô làm đại diện cho "các màu này trông giống nhau đến đâu" (so khớp màu láng giềng gần nhất, phân cụm màu k-means, ngưỡng trừ nền) đều thừa hưởng cùng sự méo mó này; chuyển sang khoảng cách Lab là một cách khắc phục đơn giản, tiêu chuẩn.

### Bài tập (Exercises)
1. Lặp lại ví dụ phân đoạn HSV so với RGB với một dải chuyển sáng đồng thời làm lệch sắc độ một chút (ví dụ trộn dần sang một màu khác ở một cạnh, không chỉ thay đổi độ sáng). Phân ngưỡng chỉ theo sắc độ có còn hoạt động tốt như vậy không?
2. Tăng hệ số lấy mẫu thưa chroma/luma từ 8 lên 20. Câu chuyện định tính (mất chroma ít thấy hơn mất luma) có còn đúng không, hay cuối cùng nó sẽ không còn đúng?
3. Bằng cách xem xét `c1`/`c2` trong thí nghiệm Lab, hãy tìm một cặp ví dụ có khoảng cách Lab rất *nhỏ* và một cặp có khoảng cách L\*a\*b\* rất *lớn* (dù cả hai có cùng khoảng cách RGB). Hiển thị bốn màu dưới dạng các ô mẫu và mô tả, bằng lời của bạn, tại sao cặp có khoảng cách Lab thấp trông giống nhau hơn.